In [ ]:
!pip install tensorflow==2.12.0
!pip install scikit-learn
!pip install pandas
!pip install numpy
!pip install matplotlib

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Concatenate, Embedding, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
from google.colab import files
import ast
import joblib
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive
import os
from collections import defaultdict
import math



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base_dir = '/content/drive/MyDrive/Colab Notebooks/Models/'

# Create base directory if it doesn't exist
if not os.path.exists(base_dir):
    os.makedirs(base_dir)
    print(f"Created base directory at {base_dir}")
else:
    print(f"Base directory already exists at {base_dir}")

experiment_name = 'wide_and_deep_experiment_dinner'
experiment_dir = os.path.join(base_dir, experiment_name)

if not os.path.exists(experiment_dir):
    os.makedirs(experiment_dir)
    print(f"Created experiment directory at {experiment_dir}")
else:
    print(f"Experiment directory already exists at {experiment_dir}")

In [ ]:
df = pd.read_csv('/content/drive/My Drive/Colab Notebooks/data/merged_dinnerratings_techniques.csv')
df = df.dropna()

In [ ]:

df['RecipeId'] = df['RecipeId'].astype(int)
df['AuthorId'] = df['AuthorId'].astype(int)
df['Rating'] = df['Rating'].astype(float)

In [ ]:
def parse_techniques(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) else x
    except:
        return []

df['techniques'] = df['techniques'].apply(parse_techniques)

# Expand 'techniques' into separate binary columns
techniques_df = pd.DataFrame(df['techniques'].tolist(), index=df.index)
techniques_df.columns = [f"technique_{i}" for i in range(techniques_df.shape[1])]
df = pd.concat([df, techniques_df], axis=1)
df = df.drop('techniques', axis=1)

print("Data after preprocessing:")
df.head()


In [ ]:
author_input = Input(shape=(1,), name='AuthorId')
recipe_input = Input(shape=(1,), name='RecipeId')
techniques_input = Input(shape=(techniques_df.shape[1],), name='Techniques')

# Wide part
wide_embed_dim = 1
wide_author_emb = Embedding(input_dim=df['AuthorId'].nunique()+1, output_dim=wide_embed_dim, name='wide_author_emb')(author_input)
wide_recipe_emb = Embedding(input_dim=df['RecipeId'].nunique()+1, output_dim=wide_embed_dim, name='wide_recipe_emb')(recipe_input)
wide_cross = tf.keras.layers.Multiply()([wide_author_emb, wide_recipe_emb])
wide_cross = Flatten()(wide_cross)
wide_features = Concatenate()([wide_cross, techniques_input])

# Deep part
deep_embed_dim = 32
deep_author_emb = Embedding(input_dim=df['AuthorId'].nunique()+1, output_dim=deep_embed_dim, name='deep_author_emb')(author_input)
deep_recipe_emb = Embedding(input_dim=df['RecipeId'].nunique()+1, output_dim=deep_embed_dim, name='deep_recipe_emb')(recipe_input)
deep_author_emb = Flatten()(deep_author_emb)
deep_recipe_emb = Flatten()(deep_recipe_emb)
deep_features = Concatenate()([deep_author_emb, deep_recipe_emb, techniques_input])

deep = Dense(128, activation='relu')(deep_features)
deep = Dropout(0.5)(deep)
deep = Dense(64, activation='relu')(deep)
deep = Dropout(0.5)(deep)

# Combine wide and deep parts
combined = Concatenate()([wide_features, deep])

# Output layer
output = Dense(1, activation='linear')(combined)


In [ ]:
model = Model(inputs=[author_input, recipe_input, techniques_input], outputs=output)

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

# Display the model summary
model.summary()


In [ ]:
X = {
    'AuthorId': df['AuthorId'].values,
    'RecipeId': df['RecipeId'].values,
    'Techniques': techniques_df.values
}
y = df['Rating'].values
# Split into training and validation sets
X_author_train, X_author_val, X_recipe_train, X_recipe_val, X_techniques_train, X_techniques_val, y_train, y_val = train_test_split(
    X['AuthorId'],
    X['RecipeId'],
    X['Techniques'],
    y,
    test_size=0.2,
    random_state=42
)


X_train = {
    'AuthorId': X_author_train,
    'RecipeId': X_recipe_train,
    'Techniques': X_techniques_train
}

X_val = {
    'AuthorId': X_author_val,
    'RecipeId': X_recipe_val,
    'Techniques': X_techniques_val
}
print(f"Training samples: {X_train['AuthorId'].shape[0]}")
print(f"Validation samples: {X_val['AuthorId'].shape[0]}")


In [ ]:
checkpoint_dir = os.path.join(experiment_dir, 'checkpoints/')

if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)
    print(f"Created checkpoint directory at {checkpoint_dir}")
else:
    print(f"Checkpoint directory already exists at {checkpoint_dir}")

# Define checkpoint filepath
checkpoint_filepath = os.path.join(checkpoint_dir, 'model_epoch_{epoch:02d}_val_loss_{val_loss:.4f}.h5')

# Initialize callbacks
checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_filepath,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=10,
    mode='min',
    restore_best_weights=True,
    verbose=1
)


In [ ]:
BATCH_SIZE = 128
EPOCHS = 20

# Train the model
history = model.fit(
    x={
        'AuthorId': X_train['AuthorId'],
        'RecipeId': X_train['RecipeId'],
        'Techniques': X_train['Techniques']
    },
    y=y_train,
    validation_data=(
        {
            'AuthorId': X_val['AuthorId'],
            'RecipeId': X_val['RecipeId'],
            'Techniques': X_val['Techniques']
        },
        y_val
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
    callbacks=[checkpoint_callback, early_stopping_callback]
)


In [ ]:
final_model_save_path = os.path.join(experiment_dir, 'final_wide_and_deep_model.h5')

# Save the model
model.save(final_model_save_path)
print(f"Final model saved to {final_model_save_path}")


In [ ]:
y_pred = model.predict({
    'AuthorId': X_val['AuthorId'],
    'RecipeId': X_val['RecipeId'],
    'Techniques': X_val['Techniques']
}).flatten()

# Compute RMSE
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"Validation RMSE: {rmse:.4f}")

# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"Validation MAE: {mae:.4f}")
K=10
# Custom Accuracy within a tolerance
def custom_accuracy(y_true, y_pred, tolerance=0.5):
    return np.mean(np.abs(y_true - y_pred) <= tolerance)

print(y_pred)



In [ ]:

results_df = pd.DataFrame(columns=['AuthorId', 'RecipeId', 'TrueRating', 'PredictedRating'])

for index, author_id in enumerate(X_val['AuthorId']):
  recipe_id = X_val['RecipeId'][index]
  new_row = {'AuthorId': author_id, 'RecipeId': recipe_id, 'TrueRating': y_val[index], 'PredictedRating': y_pred[index]}
  results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)


print(results_df.head())


In [ ]:
authors = defaultdict(int)
for author_id in X_val['AuthorId']:
  authors[author_id] += 1
print(len(authors))

In [ ]:
relevant_authors = {}

for author_id, author_data in authors.items():
    # Access the author ID and data
    if author_data > 2:
      relevant_authors[author_id] = author_data


In [ ]:
breakfast_recipe_ids = df.loc[df["IsBreakfast"] == 1, "RecipeId"].unique()
breakfast_recipe_ids_set = set(breakfast_recipe_ids)

lunch_recipe_ids = df.loc[df["IsLunch"] == 1, "RecipeId"].unique()
lunch_recipe_ids_set = set(lunch_recipe_ids)

snack_recipe_ids = df.loc[df["IsSnack"] == 1, "RecipeId"].unique()
snack_recipe_ids_set = set(snack_recipe_ids)

dinner_recipe_ids = df.loc[df["IsDinner"] == 1, "RecipeId"].unique()
dinner_recipe_ids_set = set(dinner_recipe_ids)

In [ ]:
k = 10
precisions = []
for author_id, author_data in relevant_authors.items():
  selected_rows = results_df[(results_df['AuthorId'] == author_id) & (results_df['TrueRating'] > 4.0)  & (results_df['RecipeId'].isin(dinner_recipe_ids_set))]
  if len(selected_rows) < k:
    continue

  k = math.ceil(min(K, len(selected_rows)))

  selected_rows_sorted = selected_rows.sort_values(by=['TrueRating'], ascending=False)
  top_k_rows = selected_rows_sorted.head(k)
  true_recipe_ids = top_k_rows['RecipeId'].tolist()

  selected_rows_sorted = selected_rows.sort_values(by=['PredictedRating'], ascending=False)
  top_k_rows = selected_rows_sorted.head(k)
  predicted_recipe_ids = top_k_rows['RecipeId'].tolist()

  n_rel = len(set(predicted_recipe_ids) & set(true_recipe_ids))
  precision = n_rel / k
  precisions.append(precision)

avg_precision = np.mean(precisions)
print(f"Average Precision@K for K={k}: {avg_precision:.4f}")



